# Contradiction-Type Taxonomies: RealContra vs. ContraDoc

This notebook is **documentation-only**. It loads the two processed corpora, prints every type label that appears in each, reproduces both taxonomies verbatim from the source papers under `papers/`, and presents a side-by-side comparison so the conceptual overlap is visible.

It does **not** unify the two label sets into a single column. The original labels stay on their respective CSVs and are used as-is in downstream notebooks. See the closing markdown for the reasoning.

**Inputs (read-only here)**
- `data/processed/finding_contradictions_in_text_2008/RealContra.csv` — RealContra (de Marneffe et al., 2008)
- `data/processed/ContraDoc/ContraDoc.csv` — ContraDoc (Li et al., 2024)


In [7]:
from pathlib import Path

import pandas as pd

REAL_PATH = Path("data/processed/finding_contradictions_in_text_2008/RealContra.csv")
CONTRA_PATH = Path("data/processed/ContraDoc/ContraDoc.csv")

real_df = pd.read_csv(REAL_PATH)
contra_df = pd.read_csv(CONTRA_PATH)

print(f"RealContra: {len(real_df)} rows, columns={list(real_df.columns)}")
print(f"ContraDoc:  {len(contra_df)} rows, columns={list(contra_df.columns)}")


RealContra: 579 rows, columns=['contradiction', 'type', 't', 'h']
ContraDoc:  891 rows, columns=['id', 'contradiction', 'doc_type', 'scope', 'contra_plug', 'contra_type', 'evidence', 'ref_sentences', 'text']


## Original contradiction-type labels

Below we enumerate the raw `type` (RealContra) and `contra_type` (ContraDoc) values that appear in each corpus. RealContra carries one type per row; ContraDoc rows may carry multiple pipe-joined types, so we both report the joined value-counts and the per-token expansion.


In [8]:
real_yes = real_df[real_df["contradiction"] == "YES"]
real_types = sorted(real_yes["type"].unique())
print(f"RealContra YES rows: {len(real_yes)}; {len(real_types)} distinct surface labels:")
print(real_types)
print()
print("RealContra `type` value counts (YES + NO):")
print(real_df["type"].value_counts(dropna=False))
print()

contra_yes = contra_df[contra_df["contradiction"] == "YES"]
contra_types_expanded = contra_yes["contra_type"].str.split("|").explode()
contra_types = sorted(contra_types_expanded.unique())
print(f"ContraDoc YES rows: {len(contra_yes)}; {len(contra_types)} distinct labels (after expanding `|`):")
print(contra_types)
print()
print("ContraDoc per-token value counts (YES rows, expanded):")
print(contra_types_expanded.value_counts())


RealContra YES rows: 131; 9 distinct surface labels:
['WK', 'WK_location', 'WK_temporal', 'antonym', 'factive', 'lexical', 'negation', 'numeric', 'structure']

RealContra `type` value counts (YES + NO):
type
none           448
numeric         38
lexical         28
negation        23
WK              13
antonym         12
factive          9
structure        4
WK_temporal      3
WK_location      1
Name: count, dtype: int64

ContraDoc YES rows: 449; 9 distinct labels (after expanding `|`):
['Causal', 'Content', 'Emotion/Mood/Feeling', 'Factual', 'Negation', 'Numeric', 'Other', 'Perspective/View/Opinion', 'Relation']

ContraDoc per-token value counts (YES rows, expanded):
contra_type
Content                     288
Perspective/View/Opinion    101
Negation                     87
Emotion/Mood/Feeling         86
Numeric                      65
Factual                      54
Causal                       36
Relation                     25
Other                         1
Name: count, dtype: int6

## RealContra taxonomy (de Marneffe et al., 2008)

Source: `papers/P08-1118.pdf`, Table 1 and §3.2. The paper defines **seven primary types**; the released corpus additionally carries two finer sub-tags on `WK` (`WK_temporal`, `WK_location`), giving nine surface labels in the data. The paper groups the seven primaries into two categories by detection difficulty:

**Category 1 — overt / "easy"** (closed lexical or numeric resources usually suffice):

| Label | Definition | Example T → H |
|---|---|---|
| `antonym` | Aligned antonyms or oppositional terms in matching contexts. | "Capital punishment is a *catalyst* for more crime." → "Capital punishment is a *deterrent* to crime." |
| `negation` | Explicit negation marker (`not`, `never`, restricting prepositions, downward-monotone quantifiers) flips polarity. | "...juries and *not* judges *must* impose..." → "...*only* judges *can* impose..." |
| `numeric` | Number, date, or time mismatch between aligned mentions of the same entity / event. | "...killed *more than 50* civilians..." → "...found *28* confirmed dead thus far." |

**Category 2 — subtle / "hard"** (require deeper inference):

| Label | Definition | Example T → H |
|---|---|---|
| `factive` | Factive or modal verb in T projects content that contradicts H (e.g., *not manage to* projects the negative event). | "The bombers *had not managed to enter* the embassy." → "The bombers entered the embassy." |
| `structure` | Same predicate with swapped or remapped arguments — subject ↔ object, or different role bindings. | "Jacques *Santer succeeded* Jacques *Delors* as president..." → "*Delors succeeded Santer* in the presidency..." |
| `lexical` | Subtle non-antonym lexical contrast that requires learning incompatible phrases. | "...said Judy Sgro *did nothing wrong*..." → "...Ethics Commission *accuses* Judy Sgro." |
| `WK` | Requires external world knowledge to detect. | "*Microsoft Israel*, one of the first Microsoft branches outside the USA, was founded in 1989." → "*Microsoft* was established in 1989." (Microsoft itself was founded in 1975.) |
| `WK_temporal` | RealContra-internal sub-tag of `WK` where the conflicting facts are temporal (3 rows). |   |
| `WK_location` | RealContra-internal sub-tag of `WK` where the conflicting facts are spatial (1 row). |   |


## ContraDoc taxonomy (Li et al., 2024)

Source: `papers/2024.naacl-long.362.pdf`, Table 1 and §3.2. Each YES document is tagged with one or more of **eight** types (definitions and examples reproduced verbatim from the paper):

| Label | Definition | Example original → contradiction |
|---|---|---|
| `Negation` | Negating the original sentence. | "Zully donated her kidney." → "Zully *never* donated her kidney." |
| `Numeric` | Number mismatch or number out of scope. | "All the donors are between 20 to 45 years old." → "Lisa, one of the donors, *she is 70 years old*." |
| `Content` | Changing one or multiple attributes of an event or entity. | "Zully donated her kidney *to a stranger*." → "Zully donated her kidney *to her close friend*." |
| `Perspective/View/Opinion` | Inconsistency in one's attitude / perspective / opinion. | "The doctor *spoke highly of* the project and called it a 'breakthrough'." → "The doctor *disliked* the project, saying it had no impact at all." |
| `Emotion/Mood/Feeling` | Inconsistency in one's emotion / mood. | "The rescue team searched for the boy *worriedly*." → "The rescue team searched for the boy *happily*." |
| `Relation` | Description of two mutually exclusive relations between entities. | "Jane and Tom are *a married couple*." → "Jane is Tom's *sister*." |
| `Factual` | Need external world knowledge to confirm the contradiction. | "The road T51 was located in *New York*." → "The road T51 was located in *California*." |
| `Causal` | The effect does not match the cause. | "I slam the door." → "After I do that, the door *opens*." |

A small number of rows additionally carry the catch-all label `Other` (n = 1 in the released YES split).


## Side-by-side comparison

The two taxonomies overlap conceptually but were built for different objects (sentence-pair NLI vs. document-level self-contradiction) and use different annotation schemes (single-label per row vs. multi-label). The table below lines up labels by what they *describe*; **it is not a claim that the listed labels are equivalent.**

| Phenomenon | RealContra label | ContraDoc label | How the two differ in practice |
|---|---|---|---|
| Explicit polarity flip via a negation marker | `negation` | `Negation` | Direct match. |
| Number / date / quantity disagreement | `numeric` | `Numeric` | Direct match when both numbers are stated in text. RealContra labels a numeric-looking conflict as `WK` instead if the comparison value comes from outside the text. |
| Word- or attribute-level swap | `antonym`, `lexical` | `Content` | RealContra splits the easy antonym sub-case from harder lexical contrasts. ContraDoc's `Content` is broader — any single attribute change of an entity / event, antonymous or not. |
| Same predicate, swapped or remapped arguments | `structure` | `Relation` | RealContra `structure` is a syntactic role swap on the same verb (subj ↔ obj). ContraDoc `Relation` is a semantic claim of mutually exclusive relations between entities (married vs sister). Overlap, not identity. |
| Requires external knowledge / verifiable real-world fact | `WK`, `WK_temporal`, `WK_location` | `Factual` | RealContra `WK` = "external knowledge needed to *detect* the conflict." ContraDoc `Factual` = "the conflicting attribute is a verifiable fact in the world." Both invoke world knowledge but at different layers. |
| Factive / modal projection (verb's lexical class flips truth, no `not`) | `factive` | — | RealContra-only. |
| Subjective opinion / view about same target | — | `Perspective/View/Opinion` | ContraDoc-only. RealContra's "real-life" corpus is restricted to factual claims. |
| Subjective inner emotional state about same event | — | `Emotion/Mood/Feeling` | ContraDoc-only. |
| Cause does not match effect | — | `Causal` | ContraDoc-only. |

### Why this notebook does not produce a unified label column

Forcing equivalence (e.g., `Content` ≡ `antonym + lexical`, or `Factual` ≡ `WK`) over-claims:
- The discriminator each paper uses is different — RealContra labels by *what cue is needed to detect the conflict*, ContraDoc labels by *what kind of claim is in conflict*. A single row can plausibly be tagged differently by the two schemes.
- Several merges (`Content` ⇄ `antonym + lexical`; `Relation` ⇄ `structure`; `Factual` ⇄ `WK`) overlap rather than coincide. A reviewer can — and should — push back.
- The taxonomies were built for different objects (sentence-pair NLI vs. document-level multi-label self-contradiction). Squashing them into one column hides that.

For per-type evaluation in downstream notebooks, group on the original `type` (RealContra) and `contra_type` (ContraDoc) columns separately. Use the table above when discussing cross-dataset coverage in writeups.
